# Logistic Regression Analysis

Aggregated DRIAMS (A+B+C+D) -- Top 3 drugs

**Approaches:** A) Raw LR  B) L2 LR + threshold tuned  C) PCA + L2 LR + threshold tuned
**Preprocessing:** log1p + standardise (fit on train only)
**Splitting:** Species-stratified 70/15/15

In [ ]:
!pip install maldideepkit maldiamrkit --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted at /content/drive')
    IN_COLAB = True
except ImportError:
    print('Running locally')
    IN_COLAB = False

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from maldideepkit.base.data import fit_input_transform, apply_input_transform
from maldiamrkit.evaluation import stratified_species_drug_split
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
                              ConfusionMatrixDisplay, RocCurveDisplay)
warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
np.random.seed(SEED)

In [ ]:
if IN_COLAB:
    DATA_ROOT = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet/Processed")
else:
    DATA_ROOT = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet/Processed")
OUT_DIR = Path("./results_lr_aggregated")
OUT_DIR.mkdir(exist_ok=True)
print(f"Data: {DATA_ROOT}")
print(f"Output: {OUT_DIR.resolve()}")

In [ ]:
SITES = ["Proc_DRIAMS-A", "Proc_DRIAMS-B", "Proc_DRIAMS-C", "Proc_DRIAMS-D"]
BIN_COLS = [f"bin_{i}" for i in range(6000)]
DRUGS = ["Ciprofloxacin", "Amoxicillin-Clavulanic_acid", "Gentamicin"]

def load_aggregated_drug(drug_name):
    frames = []
    for site in SITES:
        p = DATA_ROOT / site / drug_name / "data.csv"
        if p.exists():
            df = pd.read_csv(p)
            df["site"] = site
            frames.append(df)
    if not frames:
        raise FileNotFoundError(f"No data for {drug_name}")
    df_all = pd.concat(frames, ignore_index=True)
    X = df_all[BIN_COLS].values.astype("float32")
    y = df_all["label"].values.astype(int)
    species = df_all["species"].values
    print(f"  {drug_name:35s}  {X.shape[0]:6d} samples  R={sum(y==1):5d}  S={sum(y==0):5d}  ({len(frames)} sites)")
    return X, y, species

print("Loading top 3 drugs...")
drug_data = {}
for drug in DRUGS:
    drug_data[drug] = load_aggregated_drug(drug)
print("Done.")

In [ ]:
def create_splits(X, y, species, train_size=0.70, test_size=0.15, seed=SEED):
    n = len(y)
    idx = np.arange(n)
    idx_trval, idx_test, _, _ = stratified_species_drug_split(
        idx.reshape(-1, 1), y, species=species, test_size=test_size, random_state=seed)
    idx_trval = idx_trval.flatten().astype(int)
    idx_test = idx_test.flatten().astype(int)
    X_trval, X_test = X[idx_trval], X[idx_test]
    y_trval, y_test = y[idx_trval], y[idx_test]
    sp_trval = species[idx_trval]
    val_frac = 0.15 / (1 - test_size)
    n2 = len(y_trval)
    idx2 = np.arange(n2)
    idx_train, idx_val, _, _ = stratified_species_drug_split(
        idx2.reshape(-1, 1), y_trval, species=sp_trval, test_size=val_frac, random_state=seed)
    idx_train = idx_train.flatten().astype(int)
    idx_val = idx_val.flatten().astype(int)
    X_train, X_val = X_trval[idx_train], X_trval[idx_val]
    y_train, y_val = y_trval[idx_train], y_trval[idx_val]
    return {"train": (X_train, y_train), "val": (X_val, y_val), "test": (X_test, y_test)}
splits = {}
for drug in DRUGS:
    X, y, species = drug_data[drug]
    splits[drug] = create_splits(X, y, species)
    print(f"  {drug}: train={splits[drug]['train'][0].shape[0]:5d}  val={splits[drug]['val'][0].shape[0]:5d}  test={splits[drug]['test'][0].shape[0]:5d}")

In [ ]:
preprocessed = {}
for drug in DRUGS:
    X_train = splits[drug]["train"][0]
    state = fit_input_transform(X_train, "log1p+standardize")
    pp = {}
    for part in ["train", "val", "test"]:
        X = splits[drug][part][0]
        pp[part] = (apply_input_transform(X, state), splits[drug][part][1])
    preprocessed[drug] = pp
    print(f"  {drug}: mode={state['mode']}")

---
## Drug 1: Ciprofloxacin (23,662 samples across all 4 sites)

In [ ]:
DRUG = "Ciprofloxacin"
X_train, y_train = preprocessed[DRUG]["train"]
X_val,   y_val   = preprocessed[DRUG]["val"]
X_test,  y_test  = preprocessed[DRUG]["test"]
print(f"Ciprofloxacin -- Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"  Train R={sum(y_train):d} S={len(y_train)-sum(y_train):d}")
print(f"  Test  R={sum(y_test):d} S={len(y_test)-sum(y_test):d}")

In [ ]:
# Approach A: Raw LR (no penalty)
lr_raw = LogisticRegression(penalty=None, solver="lbfgs", max_iter=5000, random_state=SEED)
lr_raw.fit(X_train, y_train)

results_a = {}
for name, X_s, y_s in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    preds = lr_raw.predict(X_s)
    proba = lr_raw.predict_proba(X_s)[:, 1]
    results_a[name] = {"Acc": accuracy_score(y_s, preds), "BalAcc": balanced_accuracy_score(y_s, preds),
                       "F1": f1_score(y_s, preds, average="macro"), "AUC": roc_auc_score(y_s, proba)}
print("A: Raw LR (no penalty)")
for split, m in results_a.items():
    print(f"  {split:6s}  Acc={m['Acc']:.4f}  BalAcc={m['BalAcc']:.4f}  F1={m['F1']:.4f}  AUC={m['AUC']:.4f}")

In [ ]:
C_grid = np.linspace(5e-5, 1e-3, 15)
thresholds = np.linspace(0.05, 0.95, 91)

In [ ]:
# Approach B: L2 LR + threshold tuned
grid = GridSearchCV(
    LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced", max_iter=5000, random_state=SEED),
    param_grid={"C": C_grid}, cv=3, scoring="balanced_accuracy", n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)
lr_l2 = grid.best_estimator_
print(f"Best C: {grid.best_params_['C']}, CV BalAcc: {grid.best_score_:.4f}")

proba_val = lr_l2.predict_proba(X_val)[:, 1]
best_t_b = thresholds[np.argmax([balanced_accuracy_score(y_val, proba_val >= t) for t in thresholds])]
preds_test_tuned = (lr_l2.predict_proba(X_test)[:, 1] >= best_t_b)
test_balacc_b = balanced_accuracy_score(y_test, preds_test_tuned)
test_auc_b = roc_auc_score(y_test, lr_l2.predict_proba(X_test)[:, 1])
print(f"Best threshold: p >= {best_t_b:.3f}")
print(f"  Test BalAcc (tuned): {test_balacc_b:.4f}  Test AUC: {test_auc_b:.4f}")

In [ ]:
# Approach C: PCA + L2 LR
scaler = StandardScaler().fit(X_train)
pca = PCA(n_components=0.94, random_state=SEED).fit(scaler.transform(X_train))
X_train_pca = pca.transform(scaler.transform(X_train))
X_val_pca   = pca.transform(scaler.transform(X_val))
X_test_pca  = pca.transform(scaler.transform(X_test))
print(f"PCA: 6000 -> {pca.n_components_} components, {pca.explained_variance_ratio_.sum()*100:.1f}% variance")

grid_pca = GridSearchCV(
    LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced", max_iter=5000, random_state=SEED),
    param_grid={"C": C_grid}, cv=3, scoring="balanced_accuracy", n_jobs=-1, verbose=1)
grid_pca.fit(X_train_pca, y_train)
lr_pca = grid_pca.best_estimator_

proba_val_c = lr_pca.predict_proba(X_val_pca)[:, 1]
best_t_c = thresholds[np.argmax([balanced_accuracy_score(y_val, proba_val_c >= t) for t in thresholds])]
preds_test_c = (lr_pca.predict_proba(X_test_pca)[:, 1] >= best_t_c)
test_balacc_c = balanced_accuracy_score(y_test, preds_test_c)
test_auc_c = roc_auc_score(y_test, lr_pca.predict_proba(X_test_pca)[:, 1])
print(f"Best C: {grid_pca.best_params_['C']}, threshold: p >= {best_t_c:.3f}")
print(f"  Test BalAcc (tuned): {test_balacc_c:.4f}  Test AUC: {test_auc_c:.4f}")

In [ ]:
# Comparison bar chart
labels = ["A: Raw", "B: L2 (tuned)", "C: PCA+L2 (tuned)"]
train_vals = [results_a["train"]["BalAcc"],
              balanced_accuracy_score(y_train, lr_l2.predict(X_train)),
              balanced_accuracy_score(y_train, lr_pca.predict(X_train_pca))]
test_vals  = [results_a["test"]["BalAcc"], test_balacc_b, test_balacc_c]
auc_vals   = [results_a["test"]["AUC"], test_auc_b, test_auc_c]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(3); w = 0.25
ax.bar(x - w, train_vals, w, label="Train BalAcc", color="#aec7e8")
ax.bar(x,      test_vals,  w, label="Test BalAcc",  color="#1f77b4")
ax.bar(x + w,  auc_vals,   w, label="Test AUC",     color="#ff7f0e")
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel("Score"); ax.set_title(f"Logistic Regression -- {DRUG} (aggregated A+B+C+D)")
ax.legend(loc="lower right"); ax.set_ylim(0, 1.05); ax.axhline(0.5, color="gray", ls="--", alpha=0.4)
for i, (tr, te) in enumerate(zip(train_vals, test_vals)):
    gap = tr - te
    if abs(gap) > 0.005:
        ax.annotate(f"gap={gap:.2f}", (i, (tr+te)/2), fontsize=7, ha="center",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85))
plt.grid(True, ls='--', lw=0.5, color='gray', alpha=0.7)
plt.tight_layout(); plt.savefig(OUT_DIR / "lr_cip_comparison.pdf")
plt.show()

In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(7, 7))
for name, clf, X_eval in [("A: Raw", lr_raw, X_test),
                           (f"B: L2 (C={grid.best_params_['C']})", lr_l2, X_test),
                           (f"C: PCA+L2 (K={pca.n_components_})", lr_pca, X_test_pca)]:
    RocCurveDisplay.from_predictions(y_test, clf.predict_proba(X_eval)[:, 1], name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.set_title(f"ROC Curves -- {DRUG} (aggregated A+B+C+D)")
plt.tight_layout(); plt.savefig(OUT_DIR / "lr_cip_roc.pdf")
plt.show()

---
## Drug 2: Amoxicillin-Clavulanic acid (19,967 samples)

In [ ]:
# Drug 2: Amoxicillin-Clavulanic acid
DRUG2 = "Amoxicillin-Clavulanic_acid"
X2_train, y2_train = preprocessed[DRUG2]["train"]
X2_val,   y2_val   = preprocessed[DRUG2]["val"]
X2_test,  y2_test  = preprocessed[DRUG2]["test"]
print(f"\nDrug 2: {DRUG2} -- Train: {X2_train.shape}, Val: {X2_val.shape}, Test: {X2_test.shape}")

# B: L2 LR
grid2 = GridSearchCV(
    LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced", max_iter=5000, random_state=SEED),
    param_grid={"C": C_grid}, cv=3, scoring="balanced_accuracy", n_jobs=-1, verbose=0)
grid2.fit(X2_train, y2_train)
lr2 = grid2.best_estimator_
proba2_val = lr2.predict_proba(X2_val)[:, 1]
best_t2 = thresholds[np.argmax([balanced_accuracy_score(y2_val, proba2_val >= t) for t in thresholds])]
preds2b = (lr2.predict_proba(X2_test)[:, 1] >= best_t2)
bal2b = balanced_accuracy_score(y2_test, preds2b)
auc2b = roc_auc_score(y2_test, lr2.predict_proba(X2_test)[:, 1])
print(f"B: L2 LR  C={grid2.best_params_['C']}  thresh={best_t2:.3f}  Test BalAcc={bal2b:.4f}  AUC={auc2b:.4f}")

# C: PCA+L2 LR
scaler2 = StandardScaler().fit(X2_train)
pca2 = PCA(n_components=0.94, random_state=SEED).fit(scaler2.transform(X2_train))
X2_train_pca = pca2.transform(scaler2.transform(X2_train))
X2_test_pca  = pca2.transform(scaler2.transform(X2_test))
X2_val_pca   = pca2.transform(scaler2.transform(X2_val))
print(f"PCA: -> {pca2.n_components_} components")
grid2_pca = GridSearchCV(
    LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced", max_iter=5000, random_state=SEED),
    param_grid={"C": C_grid}, cv=3, scoring="balanced_accuracy", n_jobs=-1, verbose=0)
grid2_pca.fit(X2_train_pca, y2_train)
lr2_pca = grid2_pca.best_estimator_
proba2c_val = lr2_pca.predict_proba(X2_val_pca)[:, 1]
best_t2c = thresholds[np.argmax([balanced_accuracy_score(y2_val, proba2c_val >= t) for t in thresholds])]
preds2c = (lr2_pca.predict_proba(X2_test_pca)[:, 1] >= best_t2c)
bal2c = balanced_accuracy_score(y2_test, preds2c)
auc2c = roc_auc_score(y2_test, lr2_pca.predict_proba(X2_test_pca)[:, 1])
print(f"C: PCA+L2  C={grid2_pca.best_params_['C']}  thresh={best_t2c:.3f}  Test BalAcc={bal2c:.4f}  AUC={auc2c:.4f}")

---
## Drug 3: Gentamicin (18,312 samples)

In [ ]:
# Drug 3: Gentamicin
DRUG3 = "Gentamicin"
X3_train, y3_train = preprocessed[DRUG3]["train"]
X3_val,   y3_val   = preprocessed[DRUG3]["val"]
X3_test,  y3_test  = preprocessed[DRUG3]["test"]
print(f"\nDrug 3: {DRUG3} -- Train: {X3_train.shape}, Val: {X3_val.shape}, Test: {X3_test.shape}")

grid3 = GridSearchCV(
    LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced", max_iter=5000, random_state=SEED),
    param_grid={"C": C_grid}, cv=3, scoring="balanced_accuracy", n_jobs=-1, verbose=0)
grid3.fit(X3_train, y3_train)
lr3 = grid3.best_estimator_
proba3_val = lr3.predict_proba(X3_val)[:, 1]
best_t3 = thresholds[np.argmax([balanced_accuracy_score(y3_val, proba3_val >= t) for t in thresholds])]
preds3b = (lr3.predict_proba(X3_test)[:, 1] >= best_t3)
bal3b = balanced_accuracy_score(y3_test, preds3b)
auc3b = roc_auc_score(y3_test, lr3.predict_proba(X3_test)[:, 1])
print(f"B: L2 LR  C={grid3.best_params_['C']}  thresh={best_t3:.3f}  Test BalAcc={bal3b:.4f}  AUC={auc3b:.4f}")

scaler3 = StandardScaler().fit(X3_train)
pca3 = PCA(n_components=0.94, random_state=SEED).fit(scaler3.transform(X3_train))
X3_train_pca = pca3.transform(scaler3.transform(X3_train))
X3_test_pca  = pca3.transform(scaler3.transform(X3_test))
X3_val_pca   = pca3.transform(scaler3.transform(X3_val))
print(f"PCA: -> {pca3.n_components_} components")
grid3_pca = GridSearchCV(
    LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced", max_iter=5000, random_state=SEED),
    param_grid={"C": C_grid}, cv=3, scoring="balanced_accuracy", n_jobs=-1, verbose=0)
grid3_pca.fit(X3_train_pca, y3_train)
lr3_pca = grid3_pca.best_estimator_
proba3c_val = lr3_pca.predict_proba(X3_val_pca)[:, 1]
best_t3c = thresholds[np.argmax([balanced_accuracy_score(y3_val, proba3c_val >= t) for t in thresholds])]
preds3c = (lr3_pca.predict_proba(X3_test_pca)[:, 1] >= best_t3c)
bal3c = balanced_accuracy_score(y3_test, preds3c)
auc3c = roc_auc_score(y3_test, lr3_pca.predict_proba(X3_test_pca)[:, 1])
print(f"C: PCA+L2  C={grid3_pca.best_params_['C']}  thresh={best_t3c:.3f}  Test BalAcc={bal3c:.4f}  AUC={auc3c:.4f}")

---
## Multi-Drug Summary

In [ ]:
# Three-drug summary
multi = pd.DataFrame({
    "Drug": [DRUG, DRUG, DRUG, DRUG2, DRUG2, DRUG3, DRUG3],
    "Approach": ["A: Raw", "B: L2 (tuned)", "C: PCA+L2 (tuned)",
                 "B: L2 (tuned)", "C: PCA+L2 (tuned)", "B: L2 (tuned)", "C: PCA+L2 (tuned)"],
    "Test BalAcc": [results_a["test"]["BalAcc"], test_balacc_b, test_balacc_c,
                    bal2b, bal2c, bal3b, bal3c],
    "Test AUC": [results_a["test"]["AUC"], test_auc_b, test_auc_c,
                 auc2b, auc2c, auc3b, auc3c],
})
print("\n=== THREE-DRUG SUMMARY (aggregated A+B+C+D) ===")
print(multi.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(7); w = 0.3
ax.bar(x - w/2, multi["Test BalAcc"], w, label="Test BalAcc", color="#1f77b4")
ax.bar(x + w/2, multi["Test AUC"], w, label="Test AUC", color="#ff7f0e")
ax.set_xticks(x)
ax.set_xticklabels([f"{r['Drug'][:10]}\n{r['Approach'][:10]}" for _, r in multi.iterrows()], fontsize=7)
ax.set_ylabel("Score"); ax.set_title("Logistic Regression -- Three-Drug Summary (A+B+C+D)")
ax.legend(); ax.set_ylim(0, 1); ax.axhline(0.5, color="gray", ls="--", alpha=0.4)
plt.grid(True, ls='--', lw=0.5, color='gray', alpha=0.7)
plt.tight_layout(); plt.savefig(OUT_DIR / "lr_three_drug_summary.pdf")
plt.show()

In [ ]:
print("\nDone. Results saved to", OUT_DIR.resolve())
for i, drug in enumerate([DRUG, DRUG2, DRUG3], 1):
    _, y, _ = drug_data[drug]
    print(f"  {i}. {drug:35s} {len(y):6d} samples  R={sum(y):5d}  S={len(y)-sum(y):5d}")